In [3]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Optional
# from stock_forecast.Korea_Market.valuation.kse_valuation_machine_v1 import psr_forecast_df
import importlib
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)
import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)
import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)
import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)


def add_repo_path():
    here = Path.cwd()
    # 현재 위치부터 상위 폴더를 훑으며 DATA 폴더가 보이는 지점 찾기
    for p in [here, *here.parents]:
        if (p / "DATA").exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return str(p)
    # 못 찾으면 로컬 고정 경로(본인 PC 경로로) 마지막 보루로 추가
    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback) and fallback not in sys.path:
        sys.path.insert(0, fallback)
    return fallback

project_path = add_repo_path()
print("Using project path:", project_path)

import calendar
import time
# from DATA.stock_invest_function import *
from DATA.stock_invest_function import *
from datetime import datetime
from dateutil.relativedelta import relativedelta
warnings.filterwarnings('ignore')


# 유틸리티 함수들
def convert_to_month_end(date_str):
    try:
        # 문자열/타입 혼용 안전 변환
        date_obj = pd.to_datetime(date_str)
        if pd.isna(date_obj):
            return None

        y, m, d = date_obj.year, date_obj.month, date_obj.day

        # 1~5일 → 전달 말일
        if 1 <= d <= 5:
            if m == 1:
                prev_y, prev_m = y - 1, 12
            else:
                prev_y, prev_m = y, m - 1
            last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
            return datetime(prev_y, prev_m, last_day_prev)

        # 그 외 → 해당월 말일
        last_day_cur = calendar.monthrange(y, m)[1]
        return datetime(y, m, last_day_cur)

    except Exception:
        return None

def process_daily_to_monthly_market_data(daily_data, ticker):
    if not daily_data:
        return pd.DataFrame()
    df = pd.DataFrame(daily_data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_data = []
    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]
        last_day_data = month_data.loc[month_data['date'].idxmax()]
        monthly_data.append({
            'ticker': ticker,
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
        })
    return pd.DataFrame(monthly_data)


def fetch_revenue_data(ticker, api_key):
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"
        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"
        if not data:
            return None, "데이터 없음"
        return data, None
    except Exception as e:
        return None, f"오류: {str(e)}"

def fetch_market_data_yearly(ticker, api_key, start_year=2010):
    all_data = []
    current_year = datetime.now().year
    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"
        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
            time.sleep(0.3)
        except Exception as e:
            continue
    return all_data if all_data else None, None

def fetch_db_revenue_data(ticker, db_info, end_date='2025-08-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, saleq
        FROM US_fundq
        WHERE ticker = '{ticker}'
        AND saleq IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['revenue_billions'] = df['saleq'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'revenue_billions']]
    except Exception as e:
        return pd.DataFrame()

def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['market_cap_billions'] = df['me'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
    except Exception as e:
        return pd.DataFrame()

def calculate_enhanced_ttm_and_psr(merged_data):
    """Calculate enhanced TTM and PSR"""
    df = merged_data.copy()

    # ✅ 날짜형으로 변환 (핵심 수정)
    df['date_month_end'] = pd.to_datetime(df['date_month_end'], errors='coerce')
    df = df.sort_values(['date_month_end']).reset_index(drop=True)
    df = df.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    # Calculate TTM from quarterly revenue
    df['revenue_ttm'] = df.groupby('ticker')['revenue_billions'].rolling(window=4, min_periods=1).sum().reset_index(0,
                                                                                                                    drop=True)
    df['revenue_ttm_billions'] = df['revenue_ttm']

    # Apply 2-month shift
    df['revenue_ttm_shift'] = df.groupby('ticker')['revenue_ttm_billions'].shift(2)

    # Calculate PSR
    df['PSR_ttm'] = df['market_cap_billions'] / df['revenue_ttm_shift']

    # Handle infinite values
    df['PSR_ttm'] = df['PSR_ttm'].replace([np.inf, -np.inf], np.nan)

    return df

def prepare_revenue_ttm(
    df: pd.DataFrame,
    revenue_key: str = "revenue_billions",
    min_periods: int = 1,   # 완전한 TTM만 원하면 4로 바꾸세요
) -> pd.DataFrame:
    """
    1) revenue 칼럼들의 NaN을 '해당 행의 revenue 평균'으로 채움
    2) 각 revenue 칼럼의 4분기 합(TTM)을 *_ttm 칼럼으로 생성 (시차 없음)
    - 그룹 기준: ticker
    - 정렬 기준: date_month_end (월말 날짜)
    """
    d = df.copy()

    # --- 키 정리 ---
    # date_month_end: index에 있으면 칼럼으로 복구
    if 'date_month_end' not in d.columns:
        d = d.reset_index().rename(columns={'index': 'date_month_end'})
    d['date_month_end'] = pd.to_datetime(d['date_month_end'])

    if 'ticker' not in d.columns:
        raise ValueError("ticker 칼럼이 필요합니다.")

    # --- revenue 칼럼 자동 탐지 ---
    rev_cols = [c for c in d.columns if revenue_key in c]
    if not rev_cols:
        raise ValueError(f"'{revenue_key}' 가 포함된 칼럼을 찾지 못했습니다.")

    # --- ticker NaN 보정 ---
    # 단일 티커면 ffill/bfill로 채움, 복수 티커면 NaN 행 제거(필요 시 정책 조정)
    uniq_tickers = d['ticker'].dropna().unique()
    if len(uniq_tickers) == 1:
        d['ticker'] = d['ticker'].ffill().bfill()
    else:
        d = d[~d['ticker'].isna()].copy()

    # --- 정렬 ---
    d = d.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    # --- NaN 보간: 행 단위 평균으로 revenue 결측치 채우기 ---
    row_mean = d[rev_cols].mean(axis=1, skipna=True)
    for c in rev_cols:
        d[c] = d[c].fillna(row_mean)

    # --- TTM 계산 (최근 4분기 합, 시차 없음) ---
    for c in rev_cols:
        ttm_col = f"{c}_ttm"
        d[ttm_col] = (
            d.groupby('ticker', group_keys=False)[c]
             .rolling(window=4, min_periods=min_periods)
             .sum()
             .reset_index(level=0, drop=True)
        )

    d = d.set_index('date_month_end')

    return d

def clean_rev_data(rev_data: pd.DataFrame) -> pd.DataFrame:
    """
    1) 'revenue' 컬럼 값이 NaN인 행 제거
    2) (calendar_year, period) 중복 행 제거 (첫 번째 행만 유지)
       - 입력 순서를 그대로 기준으로 '첫째 데이터'를 보존
    """
    required = ['revenue', 'calendar_year', 'period']
    missing = [c for c in required if c not in rev_data.columns]
    if missing:
        raise ValueError(f"필수 컬럼이 없습니다: {missing}")

    d = rev_data.copy()

    # 1) revenue NaN인 행 제거
    before = len(d)
    d = d[~d['revenue'].isna()].copy()
    removed_nan = before - len(d)

    # 2) (calendar_year, period) 중복 제거 — 첫 행 유지(현재 순서 기준)
    before2 = len(d)
    d = d.drop_duplicates(subset=['calendar_year', 'period'], keep='first').reset_index(drop=True)
    removed_dup = before2 - len(d)

    print(f"[clean_rev_data_minimal] removed rows → revenue NaN: {removed_nan}, duplicates: {removed_dup}")
    return d

def _safe_get_db_market_df():
    try:
        df = fetch_db_market_data(ticker, db_info)
        # None 이거나 길이 0이면 빈 DF 반환
        if df is None or len(df) == 0:
            return pd.DataFrame()
        return df.copy()
    except Exception as e:
        print(f"[WARN] DB 조회 중 예외 발생: {e}")
        return pd.DataFrame()

# ==========================
# 복수 ticker 처리 코드
# ==========================

# ticker 리스트 정의
# ticker_list = ['ABBV', 'ADBE', 'ADP', 'AVGO', 'BERY', 'BLD', 'BLKB', 'BMY', 'CARG',
#                  'CARR', 'CCK','CHRW','CHWY','CINT', 'CLX','CSL','EA',
#                  'ECO','EL', 'EMR','FI','FIZZ', 'GE','GMAB','HOOD',
#                  'HRL','ILMN','INCY','INMD', 'IRDM', 'KMB','LEG','MASI',
#                  'MNST', 'NGVT','NVR','OSW','PFE','PHIN', 'PLXS','QGEN',
#                  'SAM', 'SLVM','STE','STZ','TNET','TPR', 'VRRM','VRTX', 'VVV']


api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

start_date_month = '2011-03-01'
end_date_month = (pd.Timestamp.today().normalize() - pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
measurement_date = pd.Timestamp.today().strftime('%Y-%m-%d')


ticker_list = ['AAPL', 'ADI', 'APH']

# 결과 저장용 리스트 및 에러 리스트
all_valuation_results = []
error_ticker_list = []

print("\n" + "=" * 80)
print(f"복수 Ticker 처리 시작: {len(ticker_list)}개")
print("=" * 80)

for idx, ticker in enumerate(ticker_list, 1):
    print(f"\n[{idx}/{len(ticker_list)}] Processing: {ticker}")
    print("-" * 80)

    try:
        # ===== 1. FMP 매출 데이터 수집 =====
        print(f"  → FMP 매출 데이터 수집 중...")
        revenue_data, error = fetch_revenue_data(ticker, api_key)

        if revenue_data is None:
            print(f"  ✗ FMP 매출 데이터 수집 실패 - {error}")
            error_ticker_list.append({'ticker': ticker, 'error': f'FMP revenue fetch failed: {error}'})
            continue

        all_revenue_data = []
        for item in revenue_data:
            all_revenue_data.append({
                'ticker': ticker,
                'date': item.get('date', ''),
                'calendar_year': item.get('calendarYear', ''),
                'period': item.get('period', ''),
                'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
                'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
            })

        fmp_revenue_df = pd.DataFrame(all_revenue_data)
        fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
        fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
        fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)
        fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)
        print(f"  ✓ FMP 매출 데이터: {len(fmp_revenue_df)}건")

        # ===== 2. DB 매출 데이터 가져오기 =====
        db_revenue_raw = fetch_db_revenue_data(ticker, db_info)
        db_revenue_df = db_revenue_raw.loc[db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()]

        mereged_rev_data = pd.merge(fmp_revenue_df, db_revenue_df, on=['ticker', 'date_month_end'], how='outer')
        rev_data = mereged_rev_data[mereged_rev_data['date_month_end'] >= start_date_month]
        rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])
        rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)
        rev_data = clean_rev_data(rev_data)

        # ===== 3. 모델 예측 =====
        print(f"  → SARIMA 예측 중...")
        periods = 4
        sarima_df, results = sarima.run_sarima_prediction(rev_data, forecast_quarters=periods, exog_col=None)
        sarima_df = sarima_df.sort_values("date_month_end").set_index("date_month_end")

        print(f"  → LSTM 예측 중...")
        lstm_raw_df, lstm_results_4q = lstm_v2.run_lstm_revenue_prediction(rev_data, ticker=ticker, prediction_quarters=4)
        lstm_df = lstm_raw_df.drop_duplicates(subset=['revenue_billions_lstm_forecast'], keep='last')

        print(f"  → Prophet 예측 중...")
        prophet_raw_df, res_4q = prophet_v3.run_prophet_revenue_only(rev_data, ticker=ticker, prediction_quarters=4)

        print(f"  → ES 예측 중...")
        es_raw_df, res_q4 = esmod.run_es_revenue_quarterly(rev_data, ticker=ticker, prediction_quarters=4)

        # ===== 4. FMP 시가총액 데이터 수집 =====
        print(f"  → FMP 시가총액 데이터 수집 중...")
        market_data, error = fetch_market_data_yearly(ticker, api_key, start_year=2010)

        if not market_data:
            print(f"  ✗ FMP 시가총액 데이터 수집 실패")
            error_ticker_list.append({'ticker': ticker, 'error': 'FMP market data fetch failed'})
            continue

        fmp_market_df = process_daily_to_monthly_market_data(market_data, ticker).copy()
        fmp_market_df['date_month_end'] = fmp_market_df['date'].apply(convert_to_month_end)
        fmp_market_df = (fmp_market_df
                        .drop_duplicates(subset=['date_month_end'])
                        .sort_values('date_month_end')
                        .reset_index(drop=True))
        print(f"  ✓ FMP 시가총액 데이터: {len(fmp_market_df)}건")

        # ===== 5. DB 시가총액 데이터 병합 =====
        db_market_df = _safe_get_db_market_df()

        if not db_market_df.empty:
            if 'date_month_end' not in db_market_df.columns:
                if 'date' in db_market_df.columns:
                    db_market_df['date_month_end'] = db_market_df['date'].apply(convert_to_month_end)
                else:
                    db_market_df = pd.DataFrame()

        if db_market_df.empty:
            merged_market_df = fmp_market_df.copy()
            merged_market_df['market_cap_billions_from_db'] = np.nan
        else:
            if 'market_cap_billions' in db_market_df.columns:
                db_market_df_renamed = db_market_df.rename(columns={'market_cap_billions': 'market_cap_billions_from_db'})
            else:
                db_market_df_renamed = db_market_df[['date_month_end']].copy()
                db_market_df_renamed['market_cap_billions_from_db'] = np.nan

            merged_market_df = fmp_market_df.merge(
                db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
                on='date_month_end',
                how='left'
            )

        if 'market_cap_billions' not in merged_market_df.columns:
            merged_market_df['market_cap_billions'] = np.nan

        if 'market_cap_billions_from_db' not in merged_market_df.columns:
            merged_market_df['market_cap_billions_from_db'] = np.nan

        merged_market_df['market_cap_billions'] = merged_market_df['market_cap_billions'].fillna(
            merged_market_df['market_cap_billions_from_db']
        )

        merged_market_df = (merged_market_df
                           .drop_duplicates(subset=['date_month_end'])
                           .sort_values('date_month_end')
                           .reset_index(drop=True))

        # ===== 6. PSR 계산 및 예측 =====
        enhanced_merged_df = pd.merge(merged_market_df[['date_month_end', 'market_cap_billions']],
                                      rev_data, on='date_month_end', how='outer')
        market_cap_resize = enhanced_merged_df[['date_month_end', 'market_cap_billions', 'ticker', 'revenue_billions']].copy()
        market_cap_resize.dropna(subset=['market_cap_billions'], inplace=True)
        market_cap_resize.ffill(limit=2, inplace=True)
        market_cap_resize = market_cap_resize[(market_cap_resize['date_month_end'] >= start_date_month) &
                                              (market_cap_resize['date_month_end'] <= end_date_month)]
        market_cap_resize = market_cap_resize.dropna(axis=0)
        enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(market_cap_resize)

        print(f"  → PSR 예측 중...")
        psr_sarima_df, psr_12_res = sarima.run_sarima_psr_only(
            df=enhanced_merged_df_with_ttm,
            periods=12,
            target_col="PSR_ttm",
            analysis_start="2012-06-01",
            warmup_months=6,
            fill_method="interpolate",
            ic="aic"
        )

        psr_lstm_df, psr_results = lstm_v2.run_lstm_psr_prediction(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)
        psr_prophet_df, psr_res = prophet_v3.run_prophet_psr_only(enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12)
        psr_es_df, psr_res_es = esmod.run_es_psr_only(
            df=enhanced_merged_df_with_ttm,
            ticker=ticker,
            prediction_months=12,
            start_date=None
        )

        # ===== 7. Valuation 종합 =====
        sarima_resize_df = sarima_df[['ticker', 'revenue_billions_sarima_noexog']].copy()
        lstm_resize_df = lstm_df[['revenue_billions_lstm_forecast']].copy()
        prophet_resize_df = prophet_raw_df[['revenue_billions_prophet_forecast']].copy()
        es_resize_df = es_raw_df[['revenue_billions_esq_forecast']].copy()

        revenue_forecast_df = pd.concat([sarima_resize_df, lstm_resize_df, prophet_resize_df, es_resize_df], axis=1)

        psr_sarima_resiae = psr_sarima_df[['PSR_ttm_sarima_forecast']]
        psr_lstm_resiae = psr_lstm_df[['PSR_ttm_lstm_forecast']]
        psr_prophet_resiae = psr_prophet_df[['PSR_prophet_forecast_noexog']]
        psr_es_resiae = psr_es_df[['PSR_es_forecast']]

        psr_forecast_df = pd.concat([psr_sarima_resiae, psr_lstm_resiae, psr_prophet_resiae, psr_es_resiae], axis=1)

        revenue_forecast_ = prepare_revenue_ttm(revenue_forecast_df)
        revenue_forecast_ttm = revenue_forecast_.filter(like='_ttm')
        revenue_forecast_ttm['ticker'] = ticker

        valuation_df = pd.concat([revenue_forecast_ttm, psr_forecast_df], axis=1)

        valuation_filled = valuation_df.copy()
        cols_to_fill = ['ticker'] + [c for c in valuation_filled.columns if 'revenue' in c]
        valuation_filled[cols_to_fill] = valuation_filled[cols_to_fill].ffill(limit=2)

        valuation_filled['sarima_valuation'] = (
            valuation_filled['revenue_billions_sarima_noexog_ttm'] *
            valuation_filled['PSR_ttm_sarima_forecast']
        )

        valuation_filled['lstm_valuation'] = (
            valuation_filled['revenue_billions_lstm_forecast_ttm'] *
            valuation_filled['PSR_ttm_lstm_forecast']
        )

        valuation_filled['prophet_valuation'] = (
            valuation_filled['revenue_billions_prophet_forecast_ttm'] *
            valuation_filled['PSR_prophet_forecast_noexog']
        )

        valuation_filled['es_valuation'] = (
            valuation_filled['revenue_billions_esq_forecast_ttm'] *
            valuation_filled['PSR_es_forecast']
        )

        if 'date_month_end' in valuation_filled.columns:
            valuation_filled = valuation_filled.sort_values('date_month_end')
            valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index(drop=True)
        else:
            valuation_filled = valuation_filled.sort_index()
            valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index()

        # 결과 저장
        all_valuation_results.append(valuation_result)
        print(f"  ✓ {ticker} 완료!")

    except Exception as e:
        print(f"  ✗ {ticker} 처리 중 오류 발생: {str(e)}")
        error_ticker_list.append({'ticker': ticker, 'error': str(e)})
        continue

# ===== 최종 결과 병합 =====
print("\n" + "=" * 80)
print("처리 완료!")
print("=" * 80)

if all_valuation_results:
    final_valuation_df = pd.concat(all_valuation_results, axis=0, ignore_index=True)
    print(f"\n✓ 성공적으로 처리된 ticker 수: {len(all_valuation_results)}")
    print(f"✓ 최종 결과 DataFrame 크기: {final_valuation_df.shape}")
    print(f"\n최종 결과 (final_valuation_df) 샘플:")
    print(final_valuation_df.head(10))
else:
    final_valuation_df = pd.DataFrame()
    print("\n✗ 성공적으로 처리된 ticker가 없습니다.")

if error_ticker_list:
    print(f"\n✗ 오류 발생 ticker 수: {len(error_ticker_list)}")
    print("\n오류 목록:")
    error_df = pd.DataFrame(error_ticker_list)
    print(error_df)
else:
    print("\n✓ 모든 ticker 처리 성공!")

# ===== 결과 저장 (선택사항) =====
# final_valuation_df.to_csv('multi_ticker_valuation_results.csv', index=False, encoding='utf-8-sig')
# error_df.to_csv('error_ticker_list.csv', index=False, encoding='utf-8-sig')

Using project path: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy

복수 Ticker 처리 시작: 3개

[1/3] Processing: AAPL
--------------------------------------------------------------------------------
  → FMP 매출 데이터 수집 중...
  ✓ FMP 매출 데이터: 160건
[clean_rev_data_minimal] removed rows → revenue NaN: 0, duplicates: 0
  → SARIMA 예측 중...
  → LSTM 예측 중...


20:46:28 - cmdstanpy - INFO - Chain [1] start processing


  → Prophet 예측 중...


20:46:29 - cmdstanpy - INFO - Chain [1] done processing


  → ES 예측 중...
  → FMP 시가총액 데이터 수집 중...
  ✓ FMP 시가총액 데이터: 190건
  → PSR 예측 중...


...

  → Prophet 예측 중...


20:48:43 - cmdstanpy - INFO - Chain [1] done processing


  → ES 예측 중...
  → FMP 시가총액 데이터 수집 중...
  ✓ FMP 시가총액 데이터: 190건
  → PSR 예측 중...


20:49:30 - cmdstanpy - INFO - Chain [1] start processing
20:49:30 - cmdstanpy - INFO - Chain [1] done processing


[INFO] 예측 시작일: 2025-09-30 | 데이터 마지막 월: 2025-08-31
  ✓ APH 완료!

처리 완료!

✓ 성공적으로 처리된 ticker 수: 3
✓ 최종 결과 DataFrame 크기: (45, 14)

최종 결과 (final_valuation_df) 샘플:
       index  revenue_billions_sarima_noexog_ttm  \
0 2025-06-30                          408.630000   
1 2025-07-31                          408.630000   
2 2025-08-31                          408.630000   
3 2025-09-30                          414.447736   
4 2025-10-31                          414.447736   
5 2025-11-30                          414.447736   
6 2025-12-31                          420.662395   
7 2026-01-31                          420.662395   
8 2026-02-28                          420.662395   
9 2026-03-31                          429.407878   

   revenue_billions_lstm_forecast_ttm  revenue_billions_prophet_forecast_ttm  \
0                          408.630000                             408.630000   
1                          408.630000                             408.630000   
2                          40

In [4]:
from pandas.tseries.offsets import MonthEnd

def identify_revenue_columns(columns):
    rev_cols = [c for c in columns if c.startswith("revenue_billions")]
    model_map = {"sarima": [], "lstm": [], "prophet": [], "es": []}
    for c in rev_cols:
        low = c.lower()
        if "sarima" in low:
            model_map["sarima"].append(c)
        elif "lstm" in low:
            model_map["lstm"].append(c)
        elif "prophet" in low:
            model_map["prophet"].append(c)
        elif "es" in low:
            model_map["es"].append(c)
    return rev_cols, model_map

def identify_valuation_columns(columns):
    val_cols = [c for c in columns if c.endswith("_valuation")]
    model_map = {"sarima": None, "lstm": None, "prophet": None, "es": None}
    for c in val_cols:
        low = c.lower()
        if "sarima" in low:
            model_map["sarima"] = c
        elif "lstm" in low:
            model_map["lstm"] = c
        elif "prophet" in low:
            model_map["prophet"] = c
        elif "es" in low:
            model_map["es"] = c
    return val_cols, model_map

def compute_growth(series: pd.Series, start_dt: pd.Timestamp) -> dict:
    s = series.dropna()
    s = s.loc[s.index >= start_dt]
    if s.empty:
        return {"start_value": np.nan, "end_value": np.nan, "growth": np.nan}
    start_value = s.iloc[0]
    end_value = s.iloc[-1]
    if pd.isna(start_value) or start_value == 0:
        growth = np.nan
    else:
        growth = (end_value / start_value) - 1.0
    return {"start_value": start_value, "end_value": end_value, "growth": growth}

def make_growth_summaries(df: pd.DataFrame):
    """
    final_valuation_df처럼 날짜가 'index' 컬럼에 들어있는 경우를 지원.
    - 'index' → date_month_end(월말 정규화) 생성
    - ticker별로 revenue/valuation 성장률 계산
    - revenue: Sarima/LSTM/Prophet/ES + 4개 평균
    - valuation: Sarima/LSTM/Prophet/ES + 최저 제외 Top3 평균
    """
    d = df.copy()

    # 1) 날짜: 'index' 컬럼을 날짜로 파싱해서 date_month_end 생성
    if "date_month_end" not in d.columns:
        if "index" in d.columns:
            d["date_month_end"] = pd.to_datetime(d["index"], errors="coerce")
        else:
            # 혹시 진짜 pandas 인덱스에 날짜가 있는 경우도 커버
            idx_dt = pd.to_datetime(d.index, errors="coerce")
            if idx_dt.notna().any():
                d = d.reset_index().rename(columns={"index": "date_month_end"})
                d["date_month_end"] = pd.to_datetime(d["date_month_end"], errors="coerce")
            else:
                raise KeyError("날짜가 들어있는 'index' 컬럼(또는 date_month_end)을 찾을 수 없습니다.")
    # 월말 정규화
    d["date_month_end"] = (d["date_month_end"] + MonthEnd(0))
    d = d.dropna(subset=["date_month_end"])

    # 2) ticker 보강(없으면 기본값)
    if "ticker" not in d.columns:
        d["ticker"] = d.get("Ticker", d.get("symbol", "UNKNOWN"))

    # 정렬
    d = d.sort_values(["ticker", "date_month_end"]).reset_index(drop=True)

    # 3) 매출/밸류에이션 컬럼 분리
    rev_cols, rev_model_map = identify_revenue_columns(d.columns)
    val_cols, val_model_map = identify_valuation_columns(d.columns)

    # 4) 이번달 말일 시작점
    start_dt = (pd.Timestamp.today() + MonthEnd(0)).normalize()

    revenue_growth_rows = []
    valuation_growth_rows = []

    for ticker, g in d.groupby("ticker"):
        g = g.set_index("date_month_end").copy()

        # --- 매출: 모델별 성장률 ---
        rev_model_cols = {
            "sarima": rev_model_map["sarima"][0] if rev_model_map["sarima"] else None,
            "lstm": rev_model_map["lstm"][0] if rev_model_map["lstm"] else None,
            "prophet": rev_model_map["prophet"][0] if rev_model_map["prophet"] else None,
            "es": rev_model_map["es"][0] if rev_model_map["es"] else None,
        }
        for model, col in rev_model_cols.items():
            if col is None or col not in g.columns:
                continue
            m = compute_growth(g[col], start_dt)
            revenue_growth_rows.append({
                "ticker": ticker,
                "series": f"revenue_{model}",
                "start_date": start_dt.date(),
                "start_value": m["start_value"],
                "end_value": m["end_value"],
                "growth": m["growth"],
            })

        # --- 매출: 4개 평균 ---
        present_rev_cols = [c for c in rev_model_cols.values() if c and c in g.columns]
        if present_rev_cols:
            g["revenue_avg_of_4"] = g[present_rev_cols].mean(axis=1, skipna=True)
            m = compute_growth(g["revenue_avg_of_4"], start_dt)
            revenue_growth_rows.append({
                "ticker": ticker,
                "series": "revenue_avg_of_4",
                "start_date": start_dt.date(),
                "start_value": m["start_value"],
                "end_value": m["end_value"],
                "growth": m["growth"],
            })

        # --- 밸류에이션: 모델별 성장률 ---
        val_model_cols = {k: v for k, v in val_model_map.items() if v is not None and v in g.columns}
        for model, col in val_model_cols.items():
            m = compute_growth(g[col], start_dt)
            valuation_growth_rows.append({
                "ticker": ticker,
                "series": f"valuation_{model}",
                "start_date": start_dt.date(),
                "start_value": m["start_value"],
                "end_value": m["end_value"],
                "growth": m["growth"],
            })

        # --- 밸류에이션: 최저 제외 Top3 평균 ---
        present_val_cols = list(val_model_cols.values())
        if present_val_cols:
            vals = g[present_val_cols].copy()
            row_min = vals.min(axis=1)
            top3_avg = (vals.sum(axis=1) - row_min) / np.maximum(vals.count(axis=1) - 1, 1)
            g["valuation_avg_top3"] = top3_avg
            m = compute_growth(g["valuation_avg_top3"], start_dt)
            valuation_growth_rows.append({
                "ticker": ticker,
                "series": "valuation_avg_top3",
                "start_date": start_dt.date(),
                "start_value": m["start_value"],
                "end_value": m["end_value"],
                "growth": m["growth"],
            })

    revenue_growth_summary = pd.DataFrame(revenue_growth_rows)
    valuation_growth_summary = pd.DataFrame(valuation_growth_rows)
    return revenue_growth_summary, valuation_growth_summary


# final_valuation_df 를 그대로 사용
rev_summary, val_summary = make_growth_summaries(final_valuation_df)

In [5]:
def _to_long(df: pd.DataFrame, category: str) -> pd.DataFrame:
    """
    rev/val summary 공통 포맷(series 컬럼을 분해)
    - category: 'revenue' 또는 'valuation'
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=[
            "ticker","category","model","start_month_end","start_value","end_value","growth","created_at"
        ])

    out = df.copy()

    # start_date -> start_month_end 로 통일
    if "start_date" in out.columns:
        out = out.rename(columns={"start_date": "start_month_end"})

    # series: 'revenue_sarima' / 'valuation_avg_top3' 등
    out["category"] = category
    # 'revenue_' 또는 'valuation_' prefix 제거 → model
    prefix = f"{category}_"
    out["model"] = out["series"].str.replace(prefix, "", regex=False)

    # 정리
    out["start_month_end"] = pd.to_datetime(out["start_month_end"], errors="coerce")
    out["created_at"] = pd.Timestamp.utcnow()

    cols = ["ticker","category","model","start_month_end","start_value","end_value","growth","created_at"]
    return out[cols]

# 1) 테이블 보장: 존재하지 않으면 생성하고, 유니크 키(ticker,category,model,start_month_end) 확보
def ensure_table_with_unique(db_info: dict, table_name: str = "valuation_growth_long"):
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )
    create_sql = f"""
    CREATE TABLE IF NOT EXISTS `{table_name}` (
      id BIGINT AUTO_INCREMENT PRIMARY KEY,
      ticker VARCHAR(16) NOT NULL,
      category VARCHAR(16) NOT NULL,
      model VARCHAR(32) NOT NULL,
      start_month_end DATE NOT NULL,
      start_value DOUBLE NULL,
      end_value DOUBLE NULL,
      growth DOUBLE NULL,
      created_at DATETIME NULL,
      UNIQUE KEY uniq_{table_name} (ticker, category, model, start_month_end)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """
    with engine.begin() as conn:
        conn.exec_driver_sql(create_sql)
        # 만약 기존 테이블이 있고 유니크키가 없다면 추가 시도 (이미 있으면 예외 무시)
        try:
            conn.exec_driver_sql(
                f"ALTER TABLE `{table_name}` ADD UNIQUE KEY uniq_{table_name} (ticker, category, model, start_month_end);"
            )
        except Exception:
            pass
    engine.dispose()

# 2) 업서트(같은 키면 덮어쓰기)
def upsert_long_to_db(long_df: pd.DataFrame, db_info: dict, table_name: str = "valuation_growth_long"):
    if long_df is None or long_df.empty:
        return 0

    # 타입 정리
    long_df = long_df.copy()
    long_df["start_month_end"] = pd.to_datetime(long_df["start_month_end"], errors="coerce").dt.date
    long_df["created_at"] = pd.to_datetime(long_df.get("created_at", pd.Timestamp.utcnow()))

    rows = [
        (
            str(r["ticker"]),
            str(r["category"]),
            str(r["model"]),
            r["start_month_end"],                                     # DATE
            (None if pd.isna(r["start_value"]) else float(r["start_value"])),
            (None if pd.isna(r["end_value"]) else float(r["end_value"])),
            (None if pd.isna(r["growth"]) else float(r["growth"])),
            r["created_at"].to_pydatetime(),                          # DATETIME
        )
        for _, r in long_df.iterrows()
    ]

    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )
    ensure_table_with_unique(db_info, table_name)

    sql = f"""
    INSERT INTO `{table_name}`
    (ticker, category, model, start_month_end, start_value, end_value, growth, created_at)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
    ON DUPLICATE KEY UPDATE
        start_value = VALUES(start_value),
        end_value   = VALUES(end_value),
        growth      = VALUES(growth),
        created_at  = VALUES(created_at);
    """

    # executemany 로 대량 업서트
    conn = engine.raw_connection()
    try:
        cur = conn.cursor()
        cur.executemany(sql, rows)
        conn.commit()
        affected = cur.rowcount
        cur.close()
    finally:
        conn.close()
        engine.dispose()

    return affected


long_df = _to_long(val_summary, 'str')  # 이전 append 방식 대신, long_df만 받고

# 업서트 저장
# affected_rows = upsert_long_to_db(long_df, db_info, table_name="us_valuation_result")
# print("upsert affected rows:", affected_rows)

In [6]:
rev_summary

,ticker,series,start_date,start_value,end_value,growth
0,AAPL,revenue_sarima,2025-10-31,414.447736,434.687636,0.048836
1,AAPL,revenue_lstm,2025-10-31,414.140193,407.139862,-0.016903
2,AAPL,revenue_prophet,2025-10-31,405.724485,410.619405,0.012065
3,AAPL,revenue_es,2025-10-31,419.904189,432.675574,0.030415
4,AAPL,revenue_avg_of_4,2025-10-31,413.554151,421.280619,0.018683
5,ADI,revenue_sarima,2025-10-31,10.998770,12.635821,0.148839
6,ADI,revenue_lstm,2025-10-31,10.359943,9.073152,-0.124208
7,ADI,revenue_prophet,2025-10-31,10.297437,9.314061,-0.095497
8,ADI,revenue_es,2025-10-31,11.003892,13.549326,0.231321
9,ADI,revenue_avg_of_4,2025-10-31,10.665011,11.143090,0.044827


In [41]:
def _rank_by_series(summary_df: pd.DataFrame, target_series: str, *,
                    fallback_series: str = None, top_k: int = None) -> pd.DataFrame:
    """
    summary_df에서 series == target_series (없으면 fallback_series)만 필터링 후
    ticker별 최대 growth를 선택, growth 내림차순으로 정렬하여 반환.
    반환 컬럼: ['ticker', 'growth']
    """
    if 'series' not in summary_df.columns or 'growth' not in summary_df.columns or 'ticker' not in summary_df.columns:
        raise ValueError("summary_df에는 최소 ['ticker','series','growth'] 컬럼이 있어야 합니다.")

    target = target_series
    uniq_series = set(summary_df['series'])

    if target not in uniq_series:
        if fallback_series and fallback_series in uniq_series:
            target = fallback_series
        else:
            raise ValueError(f"'{target_series}' 시리즈가 없고 fallback도 사용할 수 없습니다. "
                             f"사용 가능 series 예시: {sorted(list(uniq_series))[:10]} ...")

    df = (summary_df.loc[summary_df['series'].eq(target), ['ticker', 'growth']]
                      .dropna(subset=['growth']))

    # 같은 ticker가 여러 행이면 가장 큰 growth만 남김
    df = (df.groupby('ticker', as_index=False)['growth']
            .max()
            .sort_values('growth', ascending=False)
            .reset_index(drop=True))

    if top_k:
        df = df.head(top_k)

    return df

def rank_valuation_avg_top2(val_summary: pd.DataFrame, top_k: int = None) -> pd.DataFrame:
    """
    valuation_avg_top2 기준으로 랭킹. (없으면 valuation_avg_top3로 자동 대체)
    """
    return _rank_by_series(val_summary, target_series="valuation_avg_top2",
                           fallback_series="valuation_avg_top3", top_k=top_k)

def rank_revenue_avg_of_4(rev_summary: pd.DataFrame, top_k: int = None) -> pd.DataFrame:
    """
    revenue_avg_of_4 기준으로 랭킹.
    """
    return _rank_by_series(rev_summary, target_series="revenue_avg_of_4",
                           fallback_series=None, top_k=top_k)



In [42]:
# valuation 쪽: 최저 제외 평균(Top2) 기준 랭킹
val_rank = rank_valuation_avg_top2(val_summary, top_k=50)
print(val_rank.head(20))

# revenue 쪽: 4개 평균 기준 랭킹
rev_rank = rank_revenue_avg_of_4(rev_summary, top_k=50)
print(rev_rank.head(20))

   ticker    growth
0    NGVT  0.549564
1    INCY  0.473568
2     NVR  0.451830
3    CHRW  0.418938
4    AVGO  0.327728
5    HOOD  0.299340
6    ABBV  0.297233
7      EL  0.279624
8     VVV  0.252742
9    MASI  0.239687
10    PFE  0.232294
11   PHIN  0.227813
12    TPR  0.223019
13   PLXS  0.191483
14   CARG  0.187108
15   ILMN  0.158524
16    CSL  0.158442
17    LEG  0.158392
18   BERY  0.144204
19    STZ  0.121446
   ticker    growth
0    INMD  0.160234
1    VRTX  0.106961
2    ADBE  0.100820
3    VRRM  0.098147
4    NGVT  0.091034
5    BERY  0.089331
6      EA  0.080626
7    MNST  0.080568
8    IRDM  0.079722
9    AVGO  0.077189
10   CARG  0.074140
11     FI  0.067274
12    STZ  0.064243
13   INCY  0.063906
14    BLD  0.060219
15    STE  0.059279
16   HOOD  0.058204
17    NVR  0.055388
18    CSL  0.054093
19    ADP  0.045238


In [ ]:
rev_summary